In [1]:
""" Used when checked out from Git """
import sys
import os
def add_local():
    d = os.getcwd()
    assert d[-9:] == "notebooks", "This code assume that the notebook is run out of 'notebooks'. Remove this code if not needed"
    sys.path.insert( 0, d[:-10] )
    print(f"Added '{d[:-10]}' to the import path")
add_local()


Added '/mnt/c/Users/hans/OneDrive/Python3/packages/cdxcore' to the import path


In [20]:
import importlib as imp
import cdxcore.npio as _
imp.reload(_)
import cdxcore.npsh as _
imp.reload(_)

import gc as gc
from cdxcore.npsh import create_sharedarray, attach_sharedarray, read_sharedarray, is_sharedarray, del_sharedarray_file
import numpy as np

test_name = f"Test {np.random.randint(1000000)}"

test = create_sharedarray(test_name, shape=(100,2), dtype=np.float32, raise_on_error=True, force=False, full=0 )

print("/done", test_name)

/done Test 583864


In [22]:
t2 = attach_sharedarray( test_name )
print(t2.dtype, t2.shape)

float32 (100, 2)


In [23]:
del test
gc.collect()

0

In [26]:
del t2
gc.collect()

1218

In [21]:
int(0x1000000000000).to_bytes(6,"big")

OverflowError: int too big to convert

In [22]:
0x100**6

281474976710656

In [23]:
hex(0x100**6)

'0x1000000000000'

In [24]:
0x100**3

16777216

In [38]:
import numpy as np
import weakref
from multiprocessing import shared_memory

def win_create( name : str, shape : tuple, dtype : type ):
    """
    Create a shared-memory ndarray.
    """
    dtype  = np.dtype(dtype)
    nbytes = int(np.prod(shape)) * dtype.itemsize
    shm    = shared_memory.SharedMemory(create=True, size=nbytes)
    arr    = np.ndarray(shape, dtype=dtype, buffer=shm.buf, order="C")
    def _win_finalize():
        print("_win_create._win_finalize", name)
        shm.close()
    weakref.finalize(arr, _win_finalize )
    print("created", name)
    return arr, shm.name

def win_attach( name : str, shape : tuple, dtype : type|str, read_only : bool ):
    """
    Attach to an existing shared block and get an ndarray view.
    """
    shm       = shared_memory.SharedMemory(name=name, create=False, read_only=read_only )
    buf       = shm.buf
    arr = np.ndarray( shape, dtype=dtype, buffer=buf, order="C")
    if read_only:
        arr.setflags(write=False)
    def _win_finalize():
        print("_win_attach._win_finalize")
        shm.close()
    weakref.finalize(arr, _win_finalize )
    return arr

i = np.random.randint(0x10000000)
x = win_create(f"test{i}", (10,2), np.float32 )
print(x)

created test48204282
(array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.]], dtype=float32), 'wnsm_608822d8')


_win_create._win_finalize test154942693


In [9]:
import SharedArray as _sa
_sa.map_owner

AttributeError: module 'SharedArray' has no attribute 'map_owner'